In [ ]:
import os
import time
import numpy as np
import polars as pl
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
import optuna
from sklearn.model_selection import StratifiedKFold, train_test_split


# Forzamos CPU antes de importar TensorFlow para evitar problemas de cuDNN en este entorno
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# TENSORFLOW / KERAS
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input

print('TensorFlow configurado en CPU para ejecutar la LSTM de forma estable.')

In [ ]:
# ==========================================
# 0. FUNCIÓN DE VENTANA DESLIZANTE (SLIDING WINDOW)
# ==========================================
def create_sequences(X, y, time_steps):
    """
    Transforma datos tabulares 2D (muestras, características) 
    en tensores 3D (muestras, time_steps, características) para la LSTM.
    """
    Xs, ys = [], []
    for i in range(len(X) - time_steps + 1):
        Xs.append(X[i : (i + time_steps)])
        # La etiqueta será la del ÚLTIMO paquete de la secuencia
        ys.append(y[i + time_steps - 1])
    return np.array(Xs), np.array(ys)

# ==========================================
# 1. CARGA DE DATOS 
# ==========================================

TARGET_COL = "label"

df_encoded = pl.read_csv("../../DATASETS/dataSets_Reducidos/iot-23/datos_IOT_23_preparado.csv")

# Separación de características (X) y variable objetivo (y)
feature_columns = [col for col in df_encoded.columns if col != TARGET_COL]
X = df_encoded.select(feature_columns)
y_np = df_encoded[TARGET_COL].to_numpy().astype(np.int8)
X_np = X.to_numpy()

display(X.head())

indices = np.arange(X_np.shape[0])

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y_np,
)

train_idx, val_idx = train_test_split(
    train_full_idx,
    test_size=0.2,
    random_state=42,
    stratify=y_np[train_full_idx],
)

X_full_train_np = X_np[train_full_idx]
X_test_np = X_np[test_idx]
y_full_train = y_np[train_full_idx]
y_test_np = y_np[test_idx]

X_train_np = X_np[train_idx]
X_val_np = X_np[val_idx]
y_train_np = y_np[train_idx]
y_val_np = y_np[val_idx]

print(f"Entrenamiento: {len(X_train_np):,} muestras")
print(f"Validación:    {len(X_val_np):,} muestras")
print(f"Test:          {len(X_test_np):,} muestras")
print(f"Clases en test: {np.unique(y_test_np)}")
print(f"Total muestras: {len(X_np):,}")

In [ ]:
# ==========================================
# 2. FASE OPTUNA 
# ==========================================

def objective(trial):
    # --- HIPERPARÁMETROS ESTRUCTURALES (INFERENCIA) ---
    time_steps = trial.suggest_categorical("time_steps", [3, 5, 10])
    lstm_units = trial.suggest_categorical("lstm_units", [16, 32, 64, 128])
    num_layers = trial.suggest_int("num_layers", 1, 2)
    
    # --- HIPERPARÁMETROS FIJOS (POR DEFECTO) ---
    FIXED_BATCH = 1024
    FIXED_DROPOUT = 0.2

    skf = StratifiedKFold(n_splits=2, shuffle=False) # ¡Sin shuffle!

    f1_scores = []
    latencies = []

    for fold_id, (train_idx, val_idx) in enumerate(skf.split(X_full_train_np, y_full_train), start=1):
        tf.keras.backend.clear_session()

        X_train_cv_raw, X_val_cv_raw = X_full_train_np[train_idx], X_full_train_np[val_idx]
        y_train_cv, y_val_cv = y_full_train[train_idx], y_full_train[val_idx]

        # Escalado ajustado solo con el fold de entrenamiento para evitar data leakage
        scaler = StandardScaler()
        X_train_cv = scaler.fit_transform(X_train_cv_raw)
        X_val_cv = scaler.transform(X_val_cv_raw)

        # Al optimizar 'time_steps', debemos generar las ventanas en cada ciclo
        X_train_seq, y_train_seq = create_sequences(X_train_cv, y_train_cv, time_steps)
        X_val_seq, y_val_seq = create_sequences(X_val_cv, y_val_cv, time_steps)

        # Construcción dinámica de la estructura
        model = Sequential()
        
        # Nueva forma recomendada por Keras para la entrada de datos (quita el Warning)
        model.add(Input(shape=(time_steps, X_train_seq.shape[2])))
        
        if num_layers == 1:
            # Añadimos unroll=True para saltarnos el error de CuDNN del servidor
            model.add(LSTM(lstm_units, unroll=True)) # unroll=True para evitar el error de cuDNN en CPU
        else:
            # Añadimos unroll=True en ambas capas
            model.add(LSTM(lstm_units, return_sequences=True, unroll=True))
            model.add(Dropout(FIXED_DROPOUT))
            model.add(LSTM(lstm_units, unroll=True)) 
            # Aquí hemos añadido 2 capas con el mismo número de unidades para mantener la simplicidad del espacio de búsqueda. Si quisieramos más variedad, podríamos hacer que la segunda capa tenga un número diferente de unidades, pero eso aumentaría el espacio de búsqueda.

        model.add(Dropout(FIXED_DROPOUT)) # Dropout fijo para evitar sobreajuste, especialmente con pocas capas
        model.add(Dense(1, activation='sigmoid'))

        model.compile(optimizer=Adam(), loss='binary_crossentropy')

        early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

        # Entrenamiento silencioso y rápido
        model.fit(
            X_train_seq, y_train_seq,
            validation_data=(X_val_seq, y_val_seq),
            epochs=15, 
            batch_size=FIXED_BATCH,
            callbacks=[early_stop],
            verbose=0
        )

        # Evaluación: Eficacia (F1 macro)
        y_pred_prob = model.predict(X_val_seq, batch_size=FIXED_BATCH, verbose=0)
        y_pred01 = (y_pred_prob > 0.5).astype(np.int8)
        f1_scores.append(f1_score(y_val_seq, y_pred01, average="macro"))

        # Evaluación: Eficiencia (Latencia en Inferencia)
        subset = min(20000, len(X_val_seq))
        X_lat = X_val_seq[:subset]

        rep = 3
        fold_lat = []
        for _ in range(rep):
            t0 = time.perf_counter()
            _ = model.predict(X_lat, batch_size=FIXED_BATCH, verbose=0)
            t1 = time.perf_counter()
            fold_lat.append((t1 - t0) / len(X_lat) * 1000)

        latencies.append(float(np.mean(fold_lat)))

    avg_f1 = float(np.mean(f1_scores))
    avg_lat = float(np.mean(latencies))
    
    # Guardamos los parámetros que interesan como atributos extra si queremos
    trial.set_user_attr("f1_std", float(np.std(f1_scores)))

    return avg_f1, avg_lat

study = optuna.create_study(
    directions=["maximize", "minimize"],
    study_name="tfg_ids_lstm_structural_cv"
)

print("🚀 Iniciando barrido multiobjetivo con LSTM (Solo hiperparámetros estructurales)...")
study.optimize(objective, n_trials=20) # 15 pruebas bastan al reducir el espacio de búsqueda

# ==========================================
# 3. GUARDAR RESULTADOS
# ==========================================
pareto_ids = {t.number for t in study.best_trials}

trials_data = []
for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE:
        trials_data.append({
            "time_steps": t.params["time_steps"],
            "lstm_units": t.params["lstm_units"],
            "num_layers": t.params["num_layers"],
            "f1_macro": t.values[0],
            "latency_ms": t.values[1],
            "is_pareto": t.number in pareto_ids
        })

df_results = pl.DataFrame(trials_data)
df_results.write_csv("lstm_structural_results.csv")
print("\n✅ Resultados guardados en 'lstm_structural_results.csv'")

In [ ]:
# REPRESENTAMOS LA FRONTERA DE PARETO (ADAPTADO PARA LSTM)
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

# 1. Cargar resultados
df = pl.read_csv("lstm_structural_results.csv")

plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

# 2. Dibujar todos los puntos
sns.scatterplot(
    x=df["latency_ms"].to_numpy(),
    y=df["f1_macro"].to_numpy(),
    hue=df["is_pareto"].to_numpy(),
    palette={True: "red", False: "royalblue"},
    style=df["is_pareto"].to_numpy(),
    markers={True: "D", False: "o"},
    s=120, # Un poco más grandes para que se vean bien
    alpha=0.8
)

# 3. Etiquetar SOLO los puntos de la Frontera de Pareto
pareto_points = df.filter(pl.col("is_pareto") == True)

for row in pareto_points.iter_rows(named=True):
    # Creamos una etiqueta que describa la arquitectura de la red
    label = f"TS:{int(row['time_steps'])}, U:{int(row['lstm_units'])}, L:{int(row['num_layers'])}"
    
    plt.text(
        row["latency_ms"], 
        row["f1_macro"] + 0.0002, # Un pequeño ajuste hacia arriba para que no pise el punto
        label,
        fontsize=9, 
        fontweight='bold', 
        ha='center',
        va='bottom',
        bbox=dict(facecolor='white', alpha=0.6, edgecolor='none') # Fondo sutil para legibilidad
    )

# 4. Estética de la gráfica
plt.title("Frontera de Pareto LSTM: Eficacia vs. Latencia de Inferencia", fontsize=15, pad=20)
plt.xlabel("Latencia Media (ms/muestra)", fontsize=12)
plt.ylabel("F1-Score Macro", fontsize=12)
plt.legend(title="Punto Óptimo (Pareto)", labels=["No", "Sí"])

# Opcional: Ajustar límites para que las etiquetas no se corten
plt.tight_layout()

plt.show()


In [ ]:
# AHORA SOLO REPRESENTAMOS LOS MODELOS DE LA FRONTERA DE PARETO

df = pl.read_csv("lstm_structural_results.csv")
pareto_df = df.filter(pl.col("is_pareto") == True)

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

sns.scatterplot(
    x=pareto_df["latency_ms"].to_numpy(),
    y=pareto_df["f1_macro"].to_numpy(),
    color="red",
    marker="D",
    s=120,
    alpha=0.8
)


for row in pareto_points.iter_rows(named=True):
    # Creamos una etiqueta que describa la arquitectura de la red
    label = f"TS:{int(row['time_steps'])}, U:{int(row['lstm_units'])}, L:{int(row['num_layers'])}"
    
    plt.text(
        row["latency_ms"], 
        row["f1_macro"] + 0.0002, # Un pequeño ajuste hacia arriba para que no pise el punto
        label,
        fontsize=9, 
        fontweight='bold', 
        ha='center',
        va='bottom',
        bbox=dict(facecolor='white', alpha=0.6, edgecolor='none') # Fondo sutil para legibilidad
    )

plt.title("Sólo puntos de la frontera de Pareto", fontsize=14)
plt.xlabel("Latencia (ms/muestra)", fontsize=12)
plt.ylabel("F1‑Score Macro", fontsize=12)
plt.show()

# Lista de los modelos en la frontera de Pareto
print("Modelos en la frontera de Pareto:")
display(pareto_df.select(["time_steps", "lstm_units", "num_layers", "f1_macro", "latency_ms"]))

In [ ]:
from sklearn.metrics import accuracy_score

# ==========================================
# EVALUACIÓN FINAL EN TEST (3 CANDIDATOS LSTM)
# ==========================================

# Definimos los 3 mejores según tu frontera de Pareto
candidatos = [
    {"ts": 10, "u": 32, "l": 1, "nombre": "Candidato 1"},
    {"ts": 5, "u": 128,  "l": 1, "nombre": "Candidato 2"},
    {"ts": 5,  "u": 32,  "l": 1, "nombre": "Candidato 3"},
    {"ts": 10,  "u": 64,  "l": 1, "nombre": "Candidato 4"},
]

resultados_finales = []

print("--- EVALUACIÓN FINAL SOBRE EL SET DE TEST (LSTM) ---\n")

# Aseguramos etiquetas 0/1
y_full_train01 = ((y_full_train + 1) // 2).astype(np.int8)
y_test_np01    = ((y_test_np + 1) // 2).astype(np.int8)

# Escalado global solo para la fase final train/test
scaler_final = StandardScaler()
X_full_train_scaled = scaler_final.fit_transform(X_full_train_np)
X_test_scaled = scaler_final.transform(X_test_np)

for c in candidatos:
    print(f"🚀 Entrenando Perfil: {c['nombre']} (TS={c['ts']}, Units={c['u']}, Layers={c['l']})...")

    # 1. Crear secuencias específicas para este candidato con datos escalados
    X_train_seq, y_train_seq = create_sequences(X_full_train_scaled, y_full_train, c["ts"])
    X_test_seq, y_test_seq   = create_sequences(X_test_scaled, y_test_np01, c["ts"])

    # 2. Construir arquitectura
    model = Sequential()
    model.add(Input(shape=(c["ts"], X_train_seq.shape[2])))
    
    if c["l"] == 1:
        model.add(LSTM(c["u"], unroll=True))
    else:
        model.add(LSTM(c["u"], return_sequences=True, unroll=True))
        model.add(Dropout(0.2))
        model.add(LSTM(c["u"], unroll=True))
    
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy')

    # 3. Entrenamiento final
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    model.fit(
        X_train_seq, y_train_seq,
        validation_split=0.1,
        epochs=20, 
        batch_size=1024,
        callbacks=[early_stop],
        verbose=0
    )

    # 4. Medir Latencia e Inferencia
    # Warm-up (necesario en GPU para estabilizar relojes)
    _ = model.predict(X_test_seq[:min(1000, len(X_test_seq))], verbose=0)

    t0 = time.perf_counter()
    y_pred_prob = model.predict(X_test_seq, batch_size=1024, verbose=0)
    t1 = time.perf_counter()
    
    y_pred01 = (y_pred_prob > 0.5).astype(np.int8).flatten()

    # 5. Cálculo de métricas
    tiempo_total = t1 - t0
    latencia = (tiempo_total / len(y_test_seq)) * 1000
    f1_test = f1_score(y_test_seq, y_pred01, average="macro")
    acc_test = accuracy_score(y_test_seq, y_pred01)

    resultados_finales.append({
        "Perfil": c["nombre"],
        "TS": c["ts"],
        "Units": c["u"],
        "Layers": c["l"],
        "F1_Test": float(f1_test),
        "Accuracy_Test": float(acc_test),
        "Latencia_ms": float(latencia)
    })
    print(f"✅ Finalizado: F1={f1_test:.4f} | Latencia={latencia:.6f} ms\n")

# 6. Mostrar tabla comparativa
df_final_lstm = pl.DataFrame(resultados_finales)
print("\n" + "="*70)
print("              TABLA COMPARATIVA FINAL (LSTM - TEST SET)")
print("="*70)
print(df_final_lstm)